In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from functools import reduce
import pylab as plt

from scipy import stats

from importlib import reload
import itertools
from itertools import count

import sys

In [ ]:
def plot_context():
    sns.set_context("talk", rc={"font.size":18,"axes.titlesize":18,"axes.labelsize":16,"font_scale":0.9})
image_path = '/scratch/c.c21013066/images/ukbiobank/accelerometer'
data_path = '/scratch/c.c21013066/data/ukbiobank/sample/withGP'
acc_path = '/scratch/c.c21013066/data/ukbiobank/phenotypes/accelerometer'

In [ ]:
# load acc merged file
acc = pd.read_csv(f'{acc_path}/matched_all_HCnoOsteo_genebloodrisk.csv',index_col=0)
#acc['Healthy_age'] = np.nan
#acc.loc[acc['diagnosis']=='Healthy','Healthy_age'] = 999
acc = acc.drop(columns='Unnamed: 0')
# or load HC file
#acc = pd.read_csv(f'{acc_path}/allHCnoOsteo.csv').set_index('eid')
#acc['Healthy'] = 1
# for each prodromal sign load who has it and merge info with acc
prodromal = ['UrinaryIncontinence','Constipation','ErectileDysfunction','Anxiety','RBD','Hyposmia','OrthostaticHypotension']
prodromal_all = ['UrinaryIncontinence','Constipation','ErectileDysfunction','Anxiety','RBD','Hyposmia','OrthostaticHypotension',
                'Depression']
sizes = pd.DataFrame(index=prodromal,columns=['N_cases','percent_UKBB'])
for p in prodromal_all:
    cases = pd.read_csv(f'{data_path}/{p}.csv',index_col=0)
    cases[p] = 1
    sizes.loc[p,'N_cases'] = cases.shape[0]
    sizes.loc[p,'percent_UKBB'] = 100/504736*sizes.loc[p,'N_cases']
    print(cases.shape)
    print(len(np.intersect1d(acc.index,cases.index)))
    acc = pd.merge(acc,cases[[f'{p}_age',p]],right_index=True,left_index=True,how='left',suffixes=['_diag',''])
    acc.loc[acc[p].isna(),p] = 0

In [ ]:
# add PD age
PD = pd.read_csv(f'{data_path}/noOsteo/ParkinsonDisease.csv').set_index('eid')
PD = PD.rename(columns={'ParkinsonDisease_age':'PD_age'})
acc = pd.merge(acc,PD['PD_age'],right_index=True,left_index=True,how='left')

In [ ]:
diseases = ['AllCauseDementia','AlzheimerDisease','AllCauseParkinsonism','ParkinsonDisease','Dystonia','Osteoarthritis',"Healthy"]
diseases_all = ['AllCauseDementia','AlzheimerDisease','AllCauseParkinsonism','ParkinsonDisease','Dystonia','Osteoarthritis','Depression','Healthy']

In [ ]:
for d in diseases_all:
    parkin = acc[acc['diagnosis']==d]
    if parkin.shape[0] > 0:
        print(d)
        for p in prodromal_all:
            parkin[f'{p}_prodromal'] = (parkin[f'{p}_age'] <= parkin[f'{d}_age']).astype(int)
            parkin.loc[np.logical_and(parkin[p]==1,parkin[d]==0),f'{p}_prodromal'] = 1
            parkin.loc[parkin[p]==0,f'{p}_prodromal'] = np.nan
            parkin[f'{p}_beforeacc'] = (parkin[f'{p}_age'] <= parkin['accelerometry_age']).astype(int)
            parkin.loc[parkin[p]!=1,f'{p}_beforeacc'] = 0
            parkin[f'{p}_beforePD'] = (parkin[f'{p}_age'] <= parkin['PD_age']).astype(int)
            parkin.loc[np.logical_and(parkin[p]==1,parkin['PD']==0),f'{p}_beforePD'] = 1
            parkin.loc[parkin[p]==0,f'{p}_beforePD'] = 0
            parkin[f'{p}_timeto{d}'] = parkin[f'{d}_age'] - parkin[f'{p}_age']
            parkin[f'{p}_status'] = parkin[f'{p}_prodromal'].replace([0,1,np.nan],['Diseased','Prodromal','Healthy'])
            acc.loc[np.logical_and(acc.index.isin(parkin.index),acc['diagnosis']==d),f'{p}_prodromal'] = parkin[f'{p}_prodromal']
            acc.loc[np.logical_and(acc.index.isin(parkin.index),acc['diagnosis']==d),f'{p}_beforeacc'] = parkin[f'{p}_beforeacc']
            acc.loc[np.logical_and(acc.index.isin(parkin.index),acc['diagnosis']==d),f'{p}_beforePD'] = parkin[f'{p}_beforePD']

In [ ]:
prod = pd.DataFrame(index=pd.MultiIndex.from_product([diseases,prodromal_all],names=['disease','prodromal_sign']),
                    columns=['N_prodromal','N_healthy','N_diseased','percent_cases','percent_prodromal',
                                                 'time_to_diag_prod_mean',
                                                 'time_to_diag_prod_std'])
for d in diseases:
    parkin = acc[np.logical_and(acc['diagnosis']==d,acc['Status']!='Healthy')]
    #if parkin.shape[0] > 0:
    for p in prodromal_all:
        parkin[f'{p}_prodromal'] = (parkin[f'{p}_age'] <= parkin[f'{d}_age']).astype(int)
        parkin.loc[np.logical_and(parkin[p]==1,parkin[d]==0),f'{p}_prodromal'] = 1
        parkin.loc[parkin[p]==0,f'{p}_prodromal'] = np.nan
        parkin[f'{p}_beforeacc'] = (parkin[f'{p}_age'] <= parkin['accelerometry_age']).astype(int)
        parkin.loc[parkin[p]!=1,f'{p}_beforeacc'] = np.nan
        parkin[f'{p}_beforePD'] = (parkin[f'{p}_age'] <= parkin['PD_age']).astype(int)
        parkin.loc[np.logical_and(parkin[p]==1,parkin['PD']==0),f'{p}_beforePD'] = 1
        parkin.loc[parkin[p]==0,f'{p}_beforePD'] = 0
        parkin[f'{p}_timeto{d}'] = parkin[f'{d}_age'] - parkin[f'{p}_age']
        parkin[f'{p}_status'] = parkin[f'{p}_prodromal'].replace([0,1,np.nan],['Diseased','Prodromal','Healthy'])
        prod.loc[(d,p),'N_prodromal'] = parkin[parkin[f'{p}_status']=='Prodromal'].shape[0]
        prod.loc[(d,p),'N_diseased'] = parkin[parkin[f'{p}_status']=='Diseased'].shape[0]
        prod.loc[(d,p),'N_healthy'] = parkin[parkin[f'{p}_status']=='Healthy'].shape[0]
        prod.loc[(d,p),'percent_cases'] = 100/parkin.shape[0] * (prod.loc[(d,p),'N_prodromal'] + prod.loc[(d,p),'N_diseased'])
        prod.loc[(d,p),'percent_prodromal'] = 100/parkin.shape[0] * (prod.loc[(d,p),'N_prodromal'])
        prod.loc[(d,p),f'time_to_diag_prod_mean'] = parkin.loc[parkin[f'{p}_status']=='Prodromal',f'{p}_timeto{d}'].mean()
        prod.loc[(d,p),f'time_to_diag_prod_std'] = parkin.loc[parkin[f'{p}_status']=='Prodromal',f'{p}_timeto{d}'].std()

In [ ]:
# test if prodromal sign more frequent in one disease than PD
chi = pd.DataFrame(index=pd.MultiIndex.from_product([diseases,prodromal_all],names=['disease','prodromal_sign']),
                   columns=['chi2','p','dof','ex'])
for d in diseases_all:
    #if d !='ParkinsonDisease' and d != 'AllCauseParkinsonism':
        for p in prodromal_all:
            cont = pd.DataFrame(index=[d,'PD'],columns=['Healthy',p])
            cont.loc[d,'Healthy'] = prod.loc[(d,p),'N_healthy']
            cont.loc[d,p] = prod.loc[(d,p),'N_prodromal']
            cont.loc['PD','Healthy'] = prod.loc[('ParkinsonDisease',p),'N_healthy']
            cont.loc['PD',p] = prod.loc[('ParkinsonDisease',p),'N_prodromal']
            try:
                chi.loc[(d,p),:] = stats.chi2_contingency(cont)
            except:
                print('zero element for ',d,' ',p)
                print(cont)
chi = chi.dropna(axis='rows',how='all')

In [ ]:
fig = plt.figure(figsize=(20,5))
plot_context()
sns.heatmap(chi['chi2'].unstack().astype(float),annot=chi['p'].unstack(),cmap='coolwarm',center=0,fmt='.1e')

In [ ]:
# do PD more often have multiple prodromal signs?
prod_col = [f'{p}_beforePD' for p in prodromal]
acc['N_signs'] = acc[prodromal_all].sum(axis=1)
acc['N_prodromalsigns'] = acc[prod_col].sum(axis=1)

In [ ]:
fig = plt.figure(figsize=(12,7))
plot_context()
sns.barplot(data=acc[acc['Status']!='Healthy'],y='diagnosis',x='N_prodromalsigns',color='k')

In [ ]:
prod

In [ ]:
acc[np.logical_and(acc['diagnosis']=='ParkinsonDisease',acc['RBD']==1)].filter(regex='_age')

In [ ]:
dfs = pd.read_csv('/scratch/c.c21013066/data/ukbiobank/phenotypes/accelerometer/matched_all_HCnoOsteo_prodromalsigns.csv',index_col=0)

In [ ]:
acc.to_csv(f'{acc_path}/matched_all_HCnoOsteo_genebloodrisk_prodromalsigns.csv')